In [58]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [50]:
data = pd.read_csv("Rain-in-Australia/weatherAUS.csv")
df = data.copy()

In [51]:
df = df.dropna(subset=['RainTomorrow'])

In [52]:
df.head(3)

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No


In [53]:
df.info()

<class 'pandas.DataFrame'>
Index: 142193 entries, 0 to 145458
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           142193 non-null  str    
 1   Location       142193 non-null  str    
 2   MinTemp        141556 non-null  float64
 3   MaxTemp        141871 non-null  float64
 4   Rainfall       140787 non-null  float64
 5   Evaporation    81350 non-null   float64
 6   Sunshine       74377 non-null   float64
 7   WindGustDir    132863 non-null  str    
 8   WindGustSpeed  132923 non-null  float64
 9   WindDir9am     132180 non-null  str    
 10  WindDir3pm     138415 non-null  str    
 11  WindSpeed9am   140845 non-null  float64
 12  WindSpeed3pm   139563 non-null  float64
 13  Humidity9am    140419 non-null  float64
 14  Humidity3pm    138583 non-null  float64
 15  Pressure9am    128179 non-null  float64
 16  Pressure3pm    128212 non-null  float64
 17  Cloud9am       88536 non-null   float64
 18  

In [54]:
X = df.drop("RainTomorrow",axis=1)
y = df["RainTomorrow"]

category_cols = X.select_dtypes(['object', 'str']).columns.tolist()
X[category_cols] = X[category_cols].fillna('Unknown')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [55]:
miss_data = X_train.isnull().sum()
miss_data[miss_data > 0]

MinTemp            525
MaxTemp            268
Rainfall          1182
Evaporation      48791
Sunshine         54345
WindGustSpeed     7406
WindSpeed9am      1083
WindSpeed3pm      2109
Humidity9am       1420
Humidity3pm       2913
Pressure9am      11257
Pressure3pm      11225
Cloud9am         43041
Cloud3pm         45767
Temp9am            736
Temp3pm           2206
dtype: int64

In [56]:
print(X_train.columns)
print(X_train.shape)

Index(['Date', 'Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation',
       'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm',
       'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm',
       'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am',
       'Temp3pm', 'RainToday'],
      dtype='str')
(113754, 22)


In [57]:
model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    random_seed=42,
    verbose=200,
)

model.fit(
    X_train,y_train,
    cat_features=category_cols,
    eval_set=[(X_test, y_test)],
)

0:	learn: 0.6564772	test: 0.6569046	best: 0.6569046 (0)	total: 260ms	remaining: 4m 19s
200:	learn: 0.3141099	test: 0.3141649	best: 0.3141649 (200)	total: 31.9s	remaining: 2m 6s
400:	learn: 0.2969618	test: 0.3006148	best: 0.3006148 (400)	total: 1m 2s	remaining: 1m 33s
600:	learn: 0.2869230	test: 0.2944201	best: 0.2944201 (600)	total: 1m 32s	remaining: 1m 1s
800:	learn: 0.2793637	test: 0.2908957	best: 0.2908952 (799)	total: 2m 1s	remaining: 30.1s
999:	learn: 0.2734187	test: 0.2886278	best: 0.2886278 (999)	total: 2m 29s	remaining: 0us

bestTest = 0.2886278473
bestIteration = 999



CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=200)

In [59]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Загальна точність моделі: {accuracy:.2%}")

Загальна точність моделі: 87.55%
